# Payne–Lacey sidewall scattering — a hands-on explorer

**Purpose.** Build a feel for how roughness rms (σ), correlation length (Lc), mode confinement and wavelength combine
to set sidewall-scattering loss, so we can agree on how real "scattering gets worse in the UV" is.
Everything is EMode-free: a 1D symmetric slab, TE mode (E tangential to the walls — the same orientation the TM00 mode of
the ridge presents to its sidewalls), first-order (Born) perturbation theory.

**What this is — and isn't.**
* The physics is the Lacey–Payne / Marcuse radiation-loss picture: power radiated per unit length ∝ σ² × (field intensity at the wall)
  × (roughness power spectrum sampled at the spatial frequencies that couple the guided mode into radiation), divided by guided power.
* I **derived the closed-form here myself** (derivation below) and checked limits numerically. I have **not** checked it against Payne & Lacey's
  own printed formulas (paywalled; the repo notes garbled OCR of them), so treat absolute numbers as indicative and *trends* as the point.
* It is a **slab with bulk indices**, so it overstates index contrast relative to the real ridge (effective-index contrast is lower) — absolute
  dB/cm will be higher than EMode's. Use the `N1_FIXED/N2_FIXED` overrides to try other contrasts.

**Reading list** (PDF of the first is in `shg_mode_survey/literature/`):
1. Melati, Melloni, Morichetti, *Real photonic waveguides: guiding light through imperfections*, Adv. Opt. Photon. **6**, 156 (2014) — the best readable review; preprint PDF supplied.
2. Payne & Lacey, *A theoretical analysis of scattering loss from planar optical waveguides*, Opt. Quantum Electron. **26**, 977 (1994), doi:10.1007/BF00708339 (paywalled).
3. Lacey & Payne, *Radiation loss from planar waveguides with random wall imperfections*, IEE Proc. J **137**, 282 (1990) (paywalled).
4. Marcuse, *Mode conversion caused by surface imperfections of a dielectric slab waveguide*, Bell Syst. Tech. J. **48**, 3187 (1969); Tien, Appl. Opt. **10**, 2395 (1971).

Convention warning: some papers define α as an *amplitude* coefficient, α = P_rad/(2 L P_g); here (and in the repo) α is the **power** coefficient, α = P_rad/(L P_g).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})

# ============================ PARAMETERS (edit these) ============================
SIGMA_NM  = 2.5            # sidewall roughness rms [nm]
D_NM      = 400.0          # slab (waveguide) width [nm]
LC_LIST   = [2.0, 10.0, 50.0]   # correlation lengths to compare [nm]
LAMS      = np.arange(215, 621, 10.0)   # wavelengths [nm]
REF_NM    = 450.0          # reference wavelength for the normalised plots
PSD_KIND  = 'exp'          # 'exp' (exponential autocorrelation) or 'gauss'
N1_FIXED  = None           # None = AlN n_o(lambda); or a number to hold the core index fixed
N2_FIXED  = None           # None = SiO2(lambda);   or a number for the cladding
# ================================================================================

def n_aln_o(lam_nm):       # same Sellmeier as the repo (geometry.AlN_EQ_O)
    x = lam_nm / 1000.0
    return (1 + 2.8032/(1-0.015287/x**2) + 0.36335/(1-0.036095/x**2) - 33508000/(1+367200000/x**2))**0.5

def n_sio2(lam_nm):        # Malitson
    x = lam_nm / 1000.0
    return np.sqrt(1 + .6961663*x**2/(x**2-.0684043**2) + .4079426*x**2/(x**2-.1162414**2)
                   + .8974794*x**2/(x**2-9.896161**2))

def indices(lam):
    return (N1_FIXED if N1_FIXED is not None else n_aln_o(lam),
            N2_FIXED if N2_FIXED is not None else n_sio2(lam))

# ---- roughness power spectra (Fourier transform of the autocorrelation function) ----
def psd_exp(k, sigma, Lc):     # R(u) = sigma^2 exp(-|u|/Lc)      -> 2 sigma^2 Lc / (1 + (k Lc)^2)
    return 2*sigma**2*Lc/(1+(k*Lc)**2)
def psd_gauss(k, sigma, Lc):   # R(u) = sigma^2 exp(-u^2/Lc^2)    -> sigma^2 Lc sqrt(pi) exp(-(k Lc / 2)^2)
    return sigma**2*Lc*np.sqrt(np.pi)*np.exp(-(k*Lc/2)**2)
PSD = {'exp': psd_exp, 'gauss': psd_gauss}[PSD_KIND]

# ---- fundamental TE mode of a symmetric slab ----
def slab_te0(lam_nm, d_nm, n1, n2):
    k0 = 2*np.pi/(lam_nm*1e-9); d = d_nm*1e-9
    V = k0*d/2*np.sqrt(n1**2-n2**2)
    f = lambda u: u*np.tan(u) - np.sqrt(V**2-u**2)
    u = brentq(f, 1e-9, min(V, np.pi/2)-1e-9)
    kx1 = 2*u/d; gam = 2*np.sqrt(V**2-u**2)/d
    beta = np.sqrt((n1*k0)**2-kx1**2)
    norm = d/2 + np.sin(kx1*d)/(2*kx1) + np.cos(u)**2/gam         # int |E|^2 dx, with E = cos(kx1 x) in the core
    return dict(k0=k0, d=d, beta=beta, neff=beta/k0, E0sq=np.cos(u)**2/norm, gamma=gam, V=V)

# ---- radiation-mode "standing wave" factor ----
def psi_factor(theta, k0, d, n1, n2):
    """Sum over the two incidence directions (from above / from below) of |E at the wall|^2 for a unit plane wave of the
    cladding continuum at angle theta. Equals exactly 2 for a uniform medium. It is the factor that accounts for the
    core acting as a Fabry-Perot cavity for light radiated by the wall (the repo's adaptation omits it, i.e. uses 2)."""
    kz = n2*k0*np.cos(theta); kx2 = n2*k0*np.sin(theta)
    kx1 = np.sqrt((n1*k0)**2-kz**2+0j)
    out = np.empty(len(theta)); h = d/2
    for i in range(len(theta)):
        a, b = kx2[i], kx1[i]
        e, ei = np.exp(1j*b*h), np.exp(-1j*b*h)
        M = np.array([[-1, e, ei, 0], [1j*a, -1j*b*e, 1j*b*ei, 0],
                      [0, ei, e, -1], [0, 1j*b*ei, -1j*b*e, 1j*a]], dtype=complex)
        r, A, B, t = np.linalg.solve(M, np.array([1, 1j*a, 0, 0], dtype=complex))
        out[i] = abs(1+r)**2 + abs(t)**2
    return out

def setup(lam_nm, d_nm=None, nth=400):
    """Everything that depends on wavelength/geometry but NOT on sigma, Lc (so scans over those are cheap)."""
    d_nm = D_NM if d_nm is None else d_nm
    n1, n2 = indices(lam_nm)
    m = slab_te0(lam_nm, d_nm, n1, n2)
    th = np.linspace(1e-4, np.pi-1e-4, nth)
    return dict(m=m, n1=n1, n2=n2, th=th, psi=psi_factor(th, m['k0'], m['d'], n1, n2), lam=lam_nm)

def alpha(st, sigma_nm=None, Lc_nm=None, use_psi=True, psd=None, parts=False):
    """Power attenuation coefficient [1/m] from roughness on BOTH walls:
       alpha = k0^4 (n1^2-n2^2)^2 / (4 pi beta)  *  |E(d/2)|^2 / int|E|^2dx  *  Int_0^pi PSD(n2 k0 cos(th) - beta) Psi(th) dth"""
    sigma_nm = SIGMA_NM if sigma_nm is None else sigma_nm
    psd = PSD if psd is None else psd
    m, n1, n2, th = st['m'], st['n1'], st['n2'], st['th']
    w = 1.0 if use_psi else 2.0
    S = np.trapezoid(psd(n2*m['k0']*np.cos(th)-m['beta'], sigma_nm*1e-9, Lc_nm*1e-9)*(st['psi'] if use_psi else 2.0), th)
    pref = m['k0']**4*(n1**2-n2**2)**2/(4*np.pi*m['beta'])
    a = pref*m['E0sq']*S
    return (a, dict(pref=pref, E0sq=m['E0sq'], S=S, neff=m['neff'])) if parts else a

dBcm = lambda a: 10*np.log10(np.e)*a/100.0     # 1/m (power) -> dB/cm

STATES = {lam: setup(lam) for lam in sorted(set(list(LAMS) + [REF_NM, 225.0]))}
print('mode set up at', len(STATES), 'wavelengths; neff(225)=%.3f  neff(450)=%.3f' % (STATES[225.0]['m']['neff'], STATES[REF_NM]['m']['neff']))

## The formula in one place

For a symmetric slab of width *d* (indices *n₁* core, *n₂* cladding) with independent roughness *h(z)* on both walls, the first-order result is

$$\alpha \;=\; \frac{k_0^{4}\,(n_1^2-n_2^2)^2}{4\pi\,\beta}\;\;\underbrace{\frac{|E(d/2)|^2}{\int |E|^2dx}}_{\text{field at the wall}}\;\;
\underbrace{\int_0^{\pi}\!\mathrm{PSD}\big(n_2k_0\cos\theta-\beta\big)\,\Psi(\theta)\,d\theta}_{S}$$

* **PSD(k)** — power spectrum of the roughness. Exponential autocorrelation: PSD = 2σ²Lc / (1+(k·Lc)²). It is flat (= 2σ²Lc) for k ≪ 1/Lc and falls as 1/k² beyond.
* **Which k is sampled?** Light scatters from the guided mode (β = n_eff·k₀) into a radiating plane wave at angle θ in the cladding (n₂k₀cosθ). The
  roughness must supply the difference: k = β − n₂k₀cosθ, which runs from **β − n₂k₀** (θ=0) to **β + n₂k₀** (θ=π). That window is the whole story of the λ-dependence of the PSD term.
* **Ψ(θ)** — standing-wave factor for the radiated wave (=2 in uniform medium). The repo's `scattering_model.py` formula equals this expression with Ψ ≡ 2, which I verify below.

Derivation sketch: a wall bump of height *h* acts as a polarization sheet ε₀(n₁²−n₂²)·h·E(d/2)·δ(x−d/2); the 2D Green's function amplitude is 1/(2k_x); the flux is k_x|E|²/(2ωμ₀); the guided power is β∫|E|²dx/(2ωμ₀); both walls add. ωμ₀ cancels.

In [ ]:
# ---------------- sanity checks ----------------
th = np.linspace(0.05, np.pi-0.05, 7)
print('1) uniform-medium limit, Psi should be exactly 2:', np.round(psi_factor(th, 2*np.pi/450e-9, 400e-9, 1.5, 1.5), 12))

st = STATES[REF_NM]
a1 = alpha(st, sigma_nm=1.0, Lc_nm=10); a2 = alpha(st, sigma_nm=2.0, Lc_nm=10)
print('2) doubling sigma should give x4:', a2/a1)

# 3) the repo's formula (k0^3/(8 pi eta0) dn^4 * (2|E0|^2 line integral) * S_W / P_g) written out for the slab, vs this model with Psi=2
eta0 = 376.730313668; mu0 = 4e-7*np.pi
m, n1, n2 = st['m'], st['n1'], st['n2']
omega = m['k0']*299792458.0
S_W = np.trapezoid(PSD(n2*m['k0']*np.cos(st['th'])-m['beta'], SIGMA_NM*1e-9, 10e-9), st['th'])
P_rad = m['k0']**3/(8*np.pi*eta0)*(n1**2-n2**2)**2*(2*m['E0sq'])*S_W      # per unit |E|^2-normalised field; two walls
P_g = m['beta']/(2*omega*mu0)                                              # same normalisation: int|E|^2 dx = 1
print('3) repo-style formula / this model with Psi=2  (expect 1.000):', (P_rad/P_g)/alpha(st, Lc_nm=10, use_psi=False))

## 1. What does the roughness spectrum look like to the mode?

Left: PSD vs spatial frequency for three Lc. Shaded windows: the range of spatial frequencies the mode can scatter with, at 225 nm and at 450 nm.
The window moves to higher k at short wavelength (bigger β and k₀). Whether that raises or lowers scattering depends on where the PSD knee (k ≈ 1/Lc) sits relative to the window.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
k = np.logspace(0, 3.3, 600)   # 1/um
for Lc, c in zip(LC_LIST, ['#1baf7a', '#eb6834', '#2a78d6']):
    ax[0].loglog(k, PSD(k*1e6, SIGMA_NM*1e-9, Lc*1e-9)/PSD(0, SIGMA_NM*1e-9, Lc*1e-9), color=c, lw=2, label=f'Lc = {Lc:g} nm   (knee 1/Lc = {1e3/Lc:.0f} /um)')
for lam, col in ((225.0, '#7b3294'), (REF_NM, '#444444')):
    s = STATES[lam]['m']; n2 = STATES[lam]['n2']
    lo, hi = (s['beta']-n2*s['k0'])*1e-6, (s['beta']+n2*s['k0'])*1e-6
    ax[0].axvspan(lo, hi, color=col, alpha=0.13, label=f'{lam:g} nm window: k = {lo:.1f} ... {hi:.1f} /um')
ax[0].set_xlabel('roughness spatial frequency k [1/um]'); ax[0].set_ylabel('PSD(k) / PSD(0)')
ax[0].set_title('roughness spectrum and the k-window the mode samples'); ax[0].legend(fontsize=8); ax[0].set_ylim(1e-6, 2)

# right: what fraction of the PSD "area" is sampled: S integral vs Lc at 225 and 450
Lcs = np.logspace(0, 2.3, 60)
for lam, col in ((225.0, '#7b3294'), (REF_NM, '#444444')):
    S = [alpha(STATES[lam], Lc_nm=L, parts=True)[1]['S'] for L in Lcs]
    ax[1].loglog(Lcs, S, color=col, lw=2, label=f'S at {lam:g} nm')
ax[1].set_xlabel('correlation length Lc [nm]'); ax[1].set_ylabel('angular PSD integral S  [m^3]'); ax[1].legend()
ax[1].set_title('the PSD-integral factor alone')
plt.tight_layout(); plt.show()

## 2. Scattering vs wavelength at fixed geometry

Top row: absolute dB/cm (with the radiation-mode factor Ψ; dashed = repo-style Ψ≡2). Bottom left: normalised to the reference wavelength with 1/λ², 1/λ³, 1/λ⁴ guides.
Bottom right: the effective exponent p(λ) defined by α(λ)/α(ref) = (ref/λ)ᵖ — the number that says "scattering ∝ 1/λᵖ". 1/λ⁴ would be p = 4.

**About the ripples.** The solid curves wiggle with a period of ~100 nm; that is the radiation-mode factor Ψ — in a perfectly uniform slab the core is a Fabry–Perot cavity for the light the wall radiates, and its resonances sweep through as λ changes. The dashed Ψ≡2 curves are smooth. Treat the ripple as a feature of the idealised slab (a real 3D ridge will smear it); the smooth underlying trend is what matters.

In [ ]:
cols = ['#1baf7a', '#eb6834', '#2a78d6', '#7b3294', '#999999']
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
lam_ok = np.array(LAMS)
res = {}
for Lc, c in zip(LC_LIST, cols):
    a  = np.array([dBcm(alpha(STATES[l], Lc_nm=Lc)) for l in lam_ok])
    a0 = np.array([dBcm(alpha(STATES[l], Lc_nm=Lc, use_psi=False)) for l in lam_ok])
    res[Lc] = (a, a0)
    ref = dBcm(alpha(STATES[REF_NM], Lc_nm=Lc)); ref0 = dBcm(alpha(STATES[REF_NM], Lc_nm=Lc, use_psi=False))
    ax[0].semilogy(lam_ok, a, color=c, lw=2, label=f'Lc = {Lc:g} nm'); ax[0].semilogy(lam_ok, a0, '--', color=c, lw=1)
    ax[1].semilogy(lam_ok, a/ref, color=c, lw=2, label=f'Lc = {Lc:g} nm')
    sel = lam_ok < REF_NM - 30
    ax[2].plot(lam_ok[sel], np.log(a[sel]/ref)/np.log(REF_NM/lam_ok[sel]), color=c, lw=2, label=f'Lc = {Lc:g} nm')
lg = np.linspace(lam_ok.min(), lam_ok.max(), 100)
for p, ls in ((2, ':'), (3, '-.'), (4, (0, (5, 1)))):
    ax[1].semilogy(lg, (REF_NM/lg)**p, color='#555', ls=ls, lw=1.2, label=f'1/lambda^{p}')
ax[0].set_ylabel('scattering [dB/cm]'); ax[0].set_title(f'slab d={D_NM:g} nm, sigma={SIGMA_NM:g} nm   (dashed: Psi=2)')
ax[1].set_ylabel(f'scattering / value at {REF_NM:g} nm'); ax[1].set_title('normalised, with 1/lambda^n guides')
ax[2].axhline(2, color='#555', ls=':'); ax[2].axhline(3, color='#555', ls='-.'); ax[2].axhline(4, color='#555', ls='--')
ax[2].set_ylabel(f'effective exponent p between lambda and {REF_NM:g} nm'); ax[2].set_title('effective exponent p  [alpha ~ 1/lambda^p]'); ax[2].set_ylim(-1, 5)
for a_ in ax: a_.set_xlabel('wavelength [nm]'); a_.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('225 nm / 450 nm ratio:', {Lc: round(dBcm(alpha(STATES[225.0], Lc_nm=Lc))/dBcm(alpha(STATES[REF_NM], Lc_nm=Lc)), 2) for Lc in LC_LIST})

## 3. Which factor drives the trend?

α is a product of four wavelength-dependent pieces. Each curve below is that piece divided by its value at the reference wavelength, for one Lc.
Where the PSD-integral curve and the confinement curve pull in opposite directions you see why the net trend is mild.

In [ ]:
LC_DEMO = LC_LIST[0]
fig, axs = plt.subplots(1, len(LC_LIST), figsize=(5.6*len(LC_LIST), 4.4), sharey=True)
axs = np.atleast_1d(axs)
for ax_, Lc in zip(axs, LC_LIST):
    P = {k: [] for k in ('k0^4 / beta', '|E(wall)|^2 (confinement)', 'PSD integral S', 'Psi (radiation-mode)', 'product = alpha')}
    for l in lam_ok:
        a, p = alpha(STATES[l], Lc_nm=Lc, parts=True)
        S2 = np.trapezoid(PSD(STATES[l]['n2']*STATES[l]['m']['k0']*np.cos(STATES[l]['th'])-STATES[l]['m']['beta'], SIGMA_NM*1e-9, Lc*1e-9), STATES[l]['th'])
        P['k0^4 / beta'].append(p['pref'] / ((STATES[l]['n1']**2-STATES[l]['n2']**2)**2))   # pure k0^4/(4 pi beta) piece
        P['|E(wall)|^2 (confinement)'].append(p['E0sq'])
        P['PSD integral S'].append(S2)
        P['Psi (radiation-mode)'].append(p['S']/S2)
        P['product = alpha'].append(a)
    # contrast factor (n1^2-n2^2)^2 folded into the first curve so the product closes
    for k_, v in P.items():
        v = np.array(v, float)
        if k_ == 'k0^4 / beta':
            v = v * np.array([(STATES[l]['n1']**2-STATES[l]['n2']**2)**2 for l in lam_ok])
        i = np.argmin(abs(lam_ok-REF_NM))
        ax_.semilogy(lam_ok, v/v[i], lw=2.4 if k_.startswith('product') else 1.4, color='k' if k_.startswith('product') else None, ls='-' if k_.startswith('product') else '--', label=k_.replace('k0^4 / beta', 'k0^4 (n1^2-n2^2)^2 / beta'))
    ax_.set_title(f'Lc = {Lc:g} nm'); ax_.set_xlabel('wavelength [nm]'); ax_.legend(fontsize=7.5)
axs[0].set_ylabel(f'factor / its value at {REF_NM:g} nm')
plt.tight_layout(); plt.show()

## 4. The (λ, Lc) map, and the σ² rule

Left: the ratio α(225 nm)/α(450 nm) as a function of Lc. Right: α(λ, Lc) in dB/cm. Because α ∝ σ², σ only rescales the right-hand map; it cancels from every ratio.

In [ ]:
Lcs = np.logspace(np.log10(1.0), np.log10(200.0), 40)
lams = np.array(sorted(l for l in STATES if 220 <= l <= 600))
A = np.array([[dBcm(alpha(STATES[l], Lc_nm=L)) for l in lams] for L in Lcs])
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
r = [dBcm(alpha(STATES[225.0], Lc_nm=L))/dBcm(alpha(STATES[REF_NM], Lc_nm=L)) for L in Lcs]
ax[0].semilogx(Lcs, r, lw=2.4, color='#7b3294')
for v, ls in ((1, '-'), (4, ':'), (8, '-.'), (16, '--')): ax[0].axhline(v, color='#555', lw=1, ls=ls)
ax[0].text(Lcs[-1], 4.1, 'lambda^-2 (4x)', ha='right', fontsize=8); ax[0].text(Lcs[-1], 8.2, 'lambda^-3 (8x)', ha='right', fontsize=8); ax[0].text(Lcs[-1], 16.4, 'lambda^-4 (16x)', ha='right', fontsize=8)
ax[0].set_xlabel('correlation length Lc [nm]'); ax[0].set_ylabel('alpha(225 nm) / alpha(450 nm)'); ax[0].set_title('UV / blue scattering ratio vs Lc  (independent of sigma)')
pcm = ax[1].pcolormesh(lams, Lcs, np.log10(A), shading='auto'); ax[1].set_yscale('log')
plt.colorbar(pcm, ax=ax[1], label='log10 dB/cm'); ax[1].set_xlabel('wavelength [nm]'); ax[1].set_ylabel('Lc [nm]'); ax[1].set_title(f'alpha(lambda, Lc), sigma = {SIGMA_NM:g} nm')
plt.tight_layout(); plt.show()

## 5. Experiments to try (edit the parameter cell, re-run all)

1. **Hold the mode fixed.** Set `D_NM` very large (e.g. 2000) or compare `D_NM=300/400/600`: how much of the trend came from the mode tightening at short λ?
2. **Remove dispersion.** Set `N1_FIXED=2.1, N2_FIXED=1.46`: with indices frozen, does the UV rise survive? (It isolates the pure k₀-scaling + PSD effect.)
3. **Lower contrast.** `N1_FIXED=1.9` (roughly an effective index): absolute dB/cm drops steeply (∝ (n₁²−n₂²)²).
4. **Gaussian vs exponential roughness** (`PSD_KIND='gauss'`): the Gaussian PSD has no 1/k² tail, so short-Lc UV behaviour changes — how much?
5. **Where would 1/λ⁴ come from?** Literature quoting ~1/λ⁴ is usually Rayleigh-type volume scattering or empirical totals including absorption. Is there any (σ, Lc, d) in the map above for which p(λ) approaches 3–4 over 225–450 nm? In this model p stays near 2 for Lc ≲ 10 nm and collapses toward 0 (or negative) for Lc ≳ 30 nm.

## Caveats to keep in mind
* TE-in-slab, tangential field only. The real TM00 ridge mode also has field components normal to the sidewall (extra interface term), and a 2D ridge has corner/top/bottom contributions.
* Bulk-index contrast inflates absolute loss versus the real ridge. Compare *ratios and exponents*, not absolute dB/cm, to EMode.
* This model gives p ≈ 1.9 for Lc = 2 nm over 225–450 nm, while EMode's native calculation on the actual ridge gave ≈ 1.1 — an **unresolved disagreement** between the two methods that this notebook does not explain.
* Real roughness has a spectrum (several Lc at once, plus a long-wavelength tail); a single exponential Lc is the simplest assumption.